# ToxiFusionNet on MultiBully (Google Colab)

Runtime ▸ Change runtime type ▸ **GPU** first. Upload the project zip (`toxifusionnet.zip`) and the MultiBully annotation file (`.xlsx`/`.csv`, images are not needed) using the Files panel on the left.

*This notebook has not been executed by the author of the code base (no Colab access); if a cell fails, the error message will point to the cause.*

In [ ]:
!unzip -q -o toxifusionnet.zip && cd toxifusionnet && pip -q install -e '.[explain,serve,dev]' openpyxl
%cd toxifusionnet

In [ ]:
# sanity check: the 48 offline tests should pass
!python -m pytest -q

In [ ]:
# 1) What does the real file look like?  (columns, label counts, language mix, schema draft)
RAW = '/content/Cyberbully_corrected_emotion_sentiment.xlsx - cyberbully.csv'   # <- change to your uploaded file name
!python -m toxifusionnet.cli inspect --raw "$RAW"

Check that the columns listed in `configs/schema_multibully.yaml` (Img-Text, Text-Label, Sarcasm, Harmful-Score, Target) exist in your file. If the Kaggle copy uses other names, edit that YAML only.

In [ ]:
# 2) Prepare + describe: look at the number of *sarcastic hate* posts in val/test before training
!python -m toxifusionnet.cli prepare --schema configs/schema_multibully.yaml --raw "$RAW" --out data/processed/multibully.csv
!python -m toxifusionnet.cli describe --data data/processed/multibully.csv --out results/data_report.md

In [ ]:
# 3) Full study (baselines, 8 model variants x 3 seeds, robustness, explainability, latency).  Quick first try: EXTRA='train.epochs=3' SEEDS='42'
import os
os.environ['DEVICE'] = 'cuda'
!bash scripts/run_final.sh "$RAW" google/muril-base-cased

In [ ]:
print(open('results/RESULTS.md').read())

In [ ]:
# 4) Try the trained model
!python -m toxifusionnet.cli predict --run-dir runs/multibully/full/seed42 --device cuda --text "Wah bhai wah, kya genius idea hai, great job lol"

In [ ]:
# 5) Download everything
!zip -qr results_bundle.zip results runs/multibully/full/seed42
from google.colab import files
files.download('results_bundle.zip')